# Theoretical

The Jaya algorithm is a simple and efficient population-based optimization technique that requires no algorithm-specific parameters.  
It iteratively improves candidate solutions to find the global optimum.

**Key Properties:**
- Meta-heuristic algorithm
- Capable of solving both constrained and unconstrained optimization problems
- Population-based algorithm (repeatedly modifies a population)
- Does **not require any hyperparameter**

**Limitations:**
- May require more iterations for complex problems
- Performance can depend on population size and initialization

**Flowchart:**
<div style="text-align:center">
    <img src="./assets/jaya_flowchart.png" alt="flowchart for jaya algorithm">
</div>

**Mathematically:**
$$X_{i}^{k+1} = X_{i}^{k} + r_1 (X_{best} - |x_i^k|) - r_2 (X_{worst} - |x_i^k|)$$

The initial position of the particles can be calculated using the following equation:
$$X_i^0 = X_{min} + r(X_{max} - X_{min})$$

*Note:*
- $X_{i}^{k+1}$: Updated value of $i$ at iteration $k+1$
- $X_{i}^{k}$: Current position of particle $i$ at iteration $k$
- $r_1$ and $r_2$: Random numbers between $(0, 1)$

**Key Insights**
- The term $r_1 (X_{best} - |x_i^k|)$ indicates the tendency of the solution to move closer towards the best solution
- The term $r_2 (X_{worst} - |x_i^k|)$ indicates the tendency of the solution to escape the worst solution
- The solution is accepted if $X_{i}^{k+1}$ has a better fitness value than $X_{i}^{k}$

# Implementation

## Import Libraries

Import the required libraries.

*Ensure NumPy and Pandas are installed using `pip install numpy pandas`*

In [1]:
import numpy as np
import random
import pandas as pd

## Define the Objective Function

I have used Himmelblau's function.  
However, you can define your own functions by modifying the *objective* function accordingly and appropriately.

In [2]:
def objective(population: pd.DataFrame) ->list:
    objective_functions = []
    for i in range(len(population)):
        x = population.loc[i]
        objective_value = (((x[0]**2) + x[1] - 11)**2) + ((x[0] + x[1]**2) - 7)**2
        objective_functions.append(objective_value)
    return objective_functions

## Initial Population

The initial position of the particles can be calculated using the following equation:
$$X_i^0 = X_{min} + r(X_{max} - X_{min})$$

In [ ]:
# min_vector and max_vector must be sorted
def initial_population(min_vector: np.ndarray, max_vector: np.ndarray, population_size: int) ->pd.DataFrame:
    population = []
    for i in range(population_size):
        p = []
        
        for min, max in zip(min_vector, max_vector):
            r = random.random()
            value = min + r * (max - min)
            p.append(value)
            
        population.append(p)
    
    return pd.DataFrame(population)

## Update Population

In every iteration, the solutions are updated based on:
$$X_{i}^{k+1} = X_{i}^{k} + r_1 (X_{best} - |x_i^k|) - r_2 (X_{worst} - |x_i^k|)$$

In [51]:
def update_population(population: pd.DataFrame, dim: int)  ->pd.DataFrame:
    best_x = np.array(population.loc[population['fitness'].idxmin()][0:dim])
    worst_x = np.array(population.loc[population['fitness'].idxmax()][0:dim])
    
    new_population = []
    for i in range(len(population)):
        old_x = np.array(population.loc[i][0:dim])
        
        r1 = random.random()
        r2 = random.random()
        
        new_population.append(old_x + r1 * (best_x - abs(old_x)) - r2 * (worst_x - abs(old_x)))
        
    return pd.DataFrame(new_population)

## Greedy selection

The solution is accepted if $X_{i}^{k+1}$ has a better fitness value than $X_{i}^{k}$

In [52]:
def greedy_selector(population: pd.DataFrame, new_population: pd.DataFrame) ->pd.DataFrame:
    for i in range(len(population)):
        if population.loc[i]['fitness'] > new_population.loc[i]['fitness']:
            population.loc[i] = new_population.loc[i]
    return population

## Trimming

Variables suggested by the algorithm must lie within the prescribed bounds.  
Can be achieved by a **number of different approaches**. One approach is to **trim** the variables to their respective bounds.

In [58]:
def trim(new_population: pd.DataFrame, lower_bounds: np.ndarray, upper_bounds: np.ndarray) ->pd.DataFrame:
    features = new_population.columns.values
    
    for i in range(len(new_population)):
        for j in range(len(features)):
            
            if new_population.iat[i, j]>upper_bounds[j]:
                  new_population.iat[i, j]=upper_bounds[j]
                  
            if new_population.iat[i, j]<lower_bounds[j]:
                  new_population.iat[i, j]=lower_bounds[j]
                  
    return new_population

## Looping

Jaya algorithm main loop until some stopping criteria

In [7]:
def jaya(population_size: int, generation_size: int, min_vector: list, max_vector: list):
    lower_bounds = np.array(min_vector)
    upper_bounds = np.array(max_vector)
    
    population = initial_population(lower_bounds, upper_bounds, population_size)
    population['fitness'] = objective(population)
    
    dim = len(lower_bounds)
    gen = 0
    best=[]
    
    # assume stopping criteria is a fixed number of generations, but other criteria (e.g., fitness coverage) could be used
    while (gen < generation_size):
        new_population = update_population(population, dim)
        new_population = trim(new_population, lower_bounds, upper_bounds)
        new_population['fitness'] = objective(new_population)
        population = greedy_selector(population, new_population)
        
        gen += 1
        
    best = population['fitness'].min()
    best_x = population.loc[population['fitness'].idxmin()][0:dim].tolist()
    
    return best,best_x

## Executing the code

In [60]:
population_size = 25
generation_size = 1000
lower_bounds = [-5, -5]
upper_bounds = [5, 5]

best, best_x = jaya(population_size, generation_size, lower_bounds, upper_bounds)
print('The objective function value = {}'.format(best))
print('The optimum values of variables = {}'.format(best_x))

The objective function value = 7.364174300145788e-05
The optimum values of variables = [-2.804110483185614, 3.1323018384096017]


# References

[medium - jaya implementation](https://dhiraj-p-rai.medium.com/jaya-optimization-algorithm-16da8708691b)

Highly recommend to read this article to see the usage of Jaya algorithm in the Cloud and Edge computing:  
Mishra K, Pati J, Majhi SK (2020) A dynamic load scheduling in IaaS cloud using binary JAYA algorithm. J King Saud UnivComput Inf Sci 34:4914–4930